# 🔬 Project Conclusion: Evaluating LLM Reranking Methods

## 1. Overview

This project compares four different methods for reranking code completions generated by Large Language Models (LLMs). Given a prompt (an *instruction*, $i$), the model generates 8 candidate *completions* ($c$). Our goal is to find the best method to rerank these 8 candidates to find the most correct one.

The core idea is inspired by the **CodeRSA paper**, which proposes that a simple 'literal listener' model ($P(c|i)$) is insufficient. A 'pragmatic' model, which reasons about the *intent* behind an instruction ($P(i|c)$), should perform better. We test this hypothesis and an extension of our own.
## 🎓 Citation & Acknowledgments

This project, particularly the 'CodeRSA' and 'CodeRSA_p_i' experiments, is directly inspired by and based on the work in the following paper:

> Cao, Z., Apel, S., Singla, A., & Demberg, V. (2025). *Pragmatic Reasoning improves LLM Code Generation*. arXiv preprint arXiv:2502.15835.
> 
> [**[View Paper on arXiv]**](https://arxiv.org/abs/2502.15835)

We are grateful to the original authors for their foundational work.

## 2. Setup & Data Loading

First, we'll import the necessary libraries and load the results from our four experiments. The results of all the four experiments have been stored in conclusion jsonl files for both the models.

In [2]:
import pandas as pd

## 3. The Four Experiments

We evaluate four distinct reranking methods. The primary metric for comparison will be **Pass@1** , which measures whether any of the top 3 reranked completions pass the unit tests for that task.

### Experiment 1: Random Selection (Baseline)

**Goal:** Establish a baseline for comparison. 

**Method:** For each task, we randomly select 3 out of the 8 generated completions. This score tells us the 'luck' factor and provides a floor for our other methods.

### Experiment 2: Coder (Literal Listener)

**Goal:** Evaluate the model's default, 'literal' understanding.

**Method:** We rerank completions using the model's standard likelihood, $P_{LLM}(c|i)$. This represents a 'literal listener' that picks the code `c` it thinks is most likely given the instruction `i`. This is the score used by default in most generative models.

### Experiment 3: CodeRSA (Uniform Prior)

**Goal:** Test the core hypothesis of the CodeRSA paper: pragmatic reasoning ($P(i|c)$) is better than literal listening ($P(c|i)$).

**Method:** We use the pragmatic speaker score, $RS_1$. This score assumes a uniform prior for all instructions ($P(i)$ is constant) and is calculated in log-space for numerical stability:

$$RS_1(i | c) = - \frac{\log P(c | i)}{\sum_{i' \in I} \log P(c | i')}$$

### Experiment 4: CodeRSA with $P(i)$ Prior (Our Model)

**Goal:** Improve on CodeRSA by adding a non-uniform prior for instructions, $P(i)$.

**Method:**  We test our own hypothesis: humans prefer shorter, simpler instructions. This hypothesis is based on the Principle of Least Effort, which observe that humans tend to minimize the effort required for communication, often using fewer or shorter sentences to convey information when possible. We created a prior, $P(i)$, where probability is inversely proportional to the instruction's length, reflecting this tendency. This prior is incorporated into the pragmatic speaker score:

$$Score(i | c) = - \frac{\log P(c | i) + \log P(i)}{\sum_{i' \in I} (\log P(c | i') + \log P(i'))}$$

## 4. Results & Analysis


In [3]:
ds_df = pd.read_json("./ds_conc.jsonl" , lines = True )
ds_df.index = ["Pass@1" ]
tl_df = pd.read_json("./tl_conc.jsonl" , lines = True )
tl_df.index = ["Pass@1"]

print("Deepseek results ")
print(ds_df)
print()

print("TinyLlama results")
print(tl_df)


Deepseek results 
          random     coder   CodeRSA  CodeRSA_p_i
Pass@1  0.371951  0.353659  0.371951     0.376016

TinyLlama results
          random     coder   CodeRSA  CodeRSA_p_i
Pass@1  0.079268  0.075203  0.089431     0.089431


## 5. Conclusion

Our experiments, designed to compare four code reranking methods, reveal a complex but insightful set of results. The data suggests that the effectiveness of the CodeRSA framework is not universal but is highly dependent on the base model's architecture and training.

### Key Findings

1.  **The 'Coder' (Literal Listener) Framework is Flawed:**
    For both TinyLlama and DeepSeek Coder, the `coder` method (reranking by $P(c|i)$) performed *worse* than the `random` baseline at pass@1.
    * **TinyLlama:** `coder` ($7.5\%$) vs. `random` ($7.9\%$)
    * **DeepSeek:** `coder` ($35.3\%$) vs. `random` ($37.1\%$)
    This supports our hypothesis that a simple literal listener often prefers \"degenerate\" or overly simplistic, high-probability outputs that are not necessarily correct.

2.  **CodeRSA (Uniform Prior) Shows Promise for Generalist Models:**
    With TinyLlama (a generalist chat model), the `CodeRSA` method ($8.9\%\ pass@1$) provided a clear improvement over both the `coder` and `random` baselines. This demonstrates the value of the pragmatic reasoning framework when the model has a balanced ability to generate both code and instructions.

3.  **CodeRSA Fails on Specialist Models:**
    With DeepSeek Coder (a specialist code-fine-tuned model), `CodeRSA` improved on the flawed `coder` baseline but *failed to outperform the random baseline* ($37.1\%\ \text{for CodeRSA vs } 37.1\%\ \text{for random}$). 
    * We hypothesize this is *because* DeepSeek is a specialist model. While it is excellent at code *generation*, it is likely poor at the reverse task of instruction *generation* (a key step in sampling the set of instructions $I$). This weakness in generating plausible instructions cripples the pragmatic framework, leading to no improvement over random chance.

4.  **The $P(i)$ Prior Showed No Impact:**
    Our `CodeRSA_p_i` model, which introduced a length-based prior for instructions, performed identically to the standard `CodeRSA` for TinyLlLaMa. There are several potential reasons for this:
    * **A) Simplistic Formula:** The `P(i)` formula (based on inverse length) may not be a good-enough proxy for human preference.
    * **B) Weak Signal:** The `P(i)` term itself, which only models the *instruction* distribution, may not be a strong enough signal to influence the final *completion* ranking.
    * **C) Model Scale:** The models used (especially TinyLlama) and the limited scope of the experiment may have been insufficient to detect the small differences this prior might introduce.

## 6. Future Work & Limitations

While this analysis provides initial insights, its constraints point to several critical avenues for future research.

* **Develop a More Sophisticated $P(i)$ Prior:** Our current length-based prior is a naive heuristic. We hypothesize that the true "cost" or probability of an instruction is non-monotonic: very short instructions are often too vague to convey intent, while very long instructions are inefficient. Future work should explore models for $P(i)$ that reward an optimal level of detail. This could be a function that also incorporates semantic complexity, rather than just character length, to better approximate the "cost" of an instruction.The model could further be improved by incorporating $Zipf's$  $law$ :
    ${\displaystyle \ {\mathsf {word\ frequency}}\ \propto \ {\frac {1}{\ {\mathsf {word\ rank}}\ }}~.}$

* **Incorporate a $P(c)$ Code Prior:** We, along with the original CodeRSA paper, assumed a uniform prior for code, $P(c)$, meaning all code completions were considered equally likely *a priori*. This is a significant oversimplification. A more advanced model would incorporate a $P(c)$ prior, using a code-specific language model or n-gram analysis to estimate how "common" or "idiomatic" a given code candidate is. This would complete the full pragmatic listener equation: $P(c|i) \propto P(i|c) \cdot P(c)$.

* **Investigate Data-Driven Cost Models:** The cost $C(i)$ was assumed to be constant for all the instructions ,A more robust approach to $C(i)$ would be to develop a "cost model" based on human evaluations. One potential avenue is to create a dataset of instructions classified by humans according to their "difficulty" or "ambiguity." A decoder-only model could then be trained to assign a cost to new instructions based on their semantic similarity to these classified examples. However, this approach faces two challenges:
    1.  **Cost:** Generating such a human-annotated dataset is expensive and time-consuming.
    2.  **The Vagueness Paradox:** An instruction that a human labels as "low cost" (i.e., easy to write) might be the most vague and ambiguous, providing insufficient signal for the model to generate a correct completion.

* **Validate Across Diverse Model Families:** This entire analysis was run on DeepSeek and TinyLlama. These results must be validated on other model families (e.g., Llama 3, Mistral, Claude) and sizes. It is crucial to determine if our findings—particularly the failure of CodeRSA on specialist models—are generalizable or an artifact of the specific models tested.